## 04 OLS-Modelltraining

Author: "Zhanna Davtyan"

***Hinweis:*** Führen Sie dieses Notebook aus, nachdem Sie das `03 Explorative Datenanalyse` Notebook ausgeführt haben, um die erforderlichen Metadaten zu generieren.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle
import scipy.stats as stats
import statsmodels.api as sm

from pathlib import Path
from sklearn import preprocessing as pp
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from statsmodels.api import qqplot

In [ ]:
base_dir = Path.cwd()
if base_dir.name == "notebooks":
    base_dir = base_dir.parent
    os.chdir(base_dir)

In [ ]:
df_gefiltert = pd.read_pickle('ergebnisse/verarbeitete_daten.pkl')

### Einführung und Feature-Auswahl
Das OLS-Modell untersucht den linearen Einfluss ausgewählter Features auf die log-transformierte Like-Anzahl auf Instagram. Die Modellierung folgt einem strukturierten Hypothesentest:

#### Hypothese
 - H0: Die Gesamtheit der ausgewählten Prädiktoren (`is_verified`, `account_category`, `follower_count` etc.) hat keinen linearen Einfluss auf `likes_log`.
 - H1: Mindestens einer der ausgewählten Prädiktoren hat einen signifikanten linearen Einfluss auf `likes_log`.

#### Feature-Auswahl und Preprocessing

Basierend auf der EDA wurden folgende Features ausgewählt:

- Numerische Features: `follower_count`, `comment_count`, `hashtag_count`
- Kategoriale Features: `is_verified`, `is_professional_account`, `account_category`, `is_business_account`

Numerische Features wurden mit StandardScaler standardisiert, kategoriale Features mit OneHotEncoder kodiert. Für die Trainingsdaten wurde eine 60/40-Aufteilung vorgenommen, um das Modell zu trainieren und zu validieren.

In [ ]:
# FEATURES basierend auf die Ergebnisse der Explorativen Datenanalyse (EDA) 
feature_columns_for_ols = [
    
    # numerische Features
    
    'follower_count',       # Stark signifikant (+)
    'comment_count',        # Grenzwertig signifikant (+)
    'hashtag_count',        # Stark signifikant (-)
    # 'caption_length',     # Nicht signifikant 
    # 'mention_count',      # Nicht signifikant 
    # 'media_count_in_post',# Nicht signifikant 
    # 'hour_of_day',        # Nicht signifikant 
    # 'day_of_week',        # Nicht signifikant
    # 'month'               # Nicht signifikant    
    
    # kategoriale Features
    
    'is_verified',          # Stark signifikant (+)
    'is_professional_account', # Stark signifikant (+)
    'account_category',     # Viele Kategorien signifikant (+/-)
    'is_business_account',  # Grenzwertig signifikant (-)
    # 'post_type',          # Nicht signifikant
]

existing_ols_col = [col for col in feature_columns_for_ols if col in df_gefiltert.columns]
X = df_gefiltert[existing_ols_col].copy()
y = df_gefiltert['likes_log'].copy()
print(f"Anzahl Zeilen vor NaN-Behandlung in X_ols: {len(X)}")

# Fehlende Werte in 'account_category' mit 'Unbekannt' füllen
if 'account_category' in X.columns:
    X['account_category'] = X['account_category'].fillna('Unbekannt')
else: 
    X['account_category'] = 'Unbekannt'

print(f"Anzahl Zeilen nach NaN-Behandlung in X_ols: {len(X)}")
print("\nAusgewählte und bereinigte Features für OLS-Modell (X_ols):")
X.info()

# Identifiziere numerische und kategoriale Spalten 
numerical_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'bool', 'category']).columns.tolist()

categorical_cols = sorted(list(set(categorical_cols)))
numerical_cols = sorted(list(set(numerical_cols)))

print(f"\nNumerische Features: {numerical_cols}")
print("______________________________\n")
print(f"Kategoriale Features: {categorical_cols}")



In [ ]:
# Preprocessing Pipelines
numerical_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

# Kategoriale Features werden mit OneHotEncoder verarbeitet 
categorical_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')),
])


preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_cols),
    ('categorical', categorical_pipeline, categorical_cols)
], remainder='drop')


# Aufteilung in Trainings- und Testsets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

print(f"\nGröße Trainingsdaten X (OLS): {X_train.shape}")
print(f"Größe Testdaten X (OLS): {X_test.shape}")

if X_train.empty:
    raise ValueError("X_train_ols_split ist leer nach dem Splitten. Überprüfe die Daten und Feature-Auswahl.")

In [ ]:
# Basiskategorien für alle kategorialen 
for col in categorical_cols:
    unique_values = sorted(X[col].unique())
    base_category = unique_values[0]
    print(f"Variable: {col}")
    print(f"   Basiskategorie: {base_category}")
    print(f"   Weitere Kategorien: {unique_values[1:]}")
    print("_" * 200)

Dummy-Variablen haben nur zwei Werte (0, 1), die Standardisierung führt somit zu unnatürlichen Werten, deswegen wurde hier darauf verzichtet. In der Literatur und Lehrbüchern wird meist davon abgeraten. Daher wird der StandardScaler nur auf numerische Features angewendet
    
Quelle: Vgl. Wooldridge Jeffrey (2015) Introductory Econometrics, S. 211 - 219

In [ ]:
ols_model = None

try:
    X_train_processed_ols_np = preprocessor.fit_transform(X_train)
    X_test_processed_ols_np = preprocessor.transform(X_test)
    print(f"Preprocessing abgeschlossen. Shape von X_train_processed_ols_np: {X_train_processed_ols_np.shape}")

    # Feature-Namen nach dem Preprocessing holen
    transformed_feature_names_ols = []
    if numerical_cols: # Nur wenn es numerische Spalten gibt
        transformed_feature_names_ols.extend(numerical_cols)
    if categorical_cols: # Nur wenn es kategoriale Spalten gibt
        ohe_transformer = preprocessor.named_transformers_['categorical'].named_steps['onehot']
        if hasattr(ohe_transformer, 'get_feature_names_out'):
            transformed_feature_names_ols.extend(ohe_transformer.get_feature_names_out(categorical_cols))
        else: 
            print("WARNUNG: get_feature_names_out nicht verfügbar für OHE. Feature-Namen in Summary könnten unvollständig sein.")


    num_cols_after_transform = X_train_processed_ols_np.shape[1]
    if transformed_feature_names_ols and num_cols_after_transform == len(transformed_feature_names_ols):
        current_cols_for_df = transformed_feature_names_ols
    else:
        print(f"WARNUNG: Anzahl extrahierter Namen ({len(transformed_feature_names_ols)}) passt nicht zu Spalten ({num_cols_after_transform}). Verwende generische Namen.")
        current_cols_for_df = [f"feature_{i}" for i in range(num_cols_after_transform)]
        transformed_feature_names_ols = current_cols_for_df 

    X_train_processed_ols_df = pd.DataFrame(X_train_processed_ols_np, columns=current_cols_for_df, index=X_train.index)
    X_test_processed_ols_df = pd.DataFrame(X_test_processed_ols_np, columns=current_cols_for_df, index=X_test.index)

    # Konstante für den Intercept 
    X_train_sm = sm.add_constant(X_train_processed_ols_df.reset_index(drop=True), has_constant='add')
    X_test_sm = sm.add_constant(X_test_processed_ols_df.reset_index(drop=True), has_constant='add')

    print("\nOLS-Modelltraining...")
    ols_model_obj = sm.OLS(y_train.reset_index(drop=True), X_train_sm)
    ols_model = ols_model_obj.fit() 
    print("OLS-Modelltraining abgeschlossen.")

except Exception as e:
    print(f"Fehler während des Preprocessings oder OLS-Modelltrainings: {e}")

In [ ]:
print(f"Preprocessing abgeschlossen. Shape von X_train_processed_ols_np: {X_train_processed_ols_np.shape}")

# Feature-Namen nach dem Preprocessing holen
transformed_feature_names_ols = []
if numerical_cols: # Nur wenn es numerische Spalten gibt
    transformed_feature_names_ols.extend(numerical_cols)
if categorical_cols: # Nur wenn es kategoriale Spalten gibt
    ohe_transformer = preprocessor.named_transformers_['categorical'].named_steps['onehot']
    if hasattr(ohe_transformer, 'get_feature_names_out'):
        transformed_feature_names_ols.extend(ohe_transformer.get_feature_names_out(categorical_cols))
    else: 
        print("WARNUNG: get_feature_names_out nicht verfügbar für OHE. Feature-Namen in Summary könnten unvollständig sein.")


num_cols_after_transform = X_train_processed_ols_np.shape[1]
if transformed_feature_names_ols and num_cols_after_transform == len(transformed_feature_names_ols):
    current_cols_for_df = transformed_feature_names_ols
else:
    print(f"WARNUNG: Anzahl extrahierter Namen ({len(transformed_feature_names_ols)}) passt nicht zu Spalten ({num_cols_after_transform}). Verwende generische Namen.")
    current_cols_for_df = [f"feature_{i}" for i in range(num_cols_after_transform)]
    transformed_feature_names_ols = current_cols_for_df 

X_train_processed_ols_df = pd.DataFrame(X_train_processed_ols_np, columns=current_cols_for_df, index=X_train.index)
X_test_processed_ols_df = pd.DataFrame(X_test_processed_ols_np, columns=current_cols_for_df, index=X_test.index)

# Konstante für den Intercept 
X_train_sm = sm.add_constant(X_train_processed_ols_df.reset_index(drop=True), has_constant='add')
X_test_sm = sm.add_constant(X_test_processed_ols_df.reset_index(drop=True), has_constant='add')

print("\nOLS-Modelltraining...")
ols_model_obj = sm.OLS(y_train.reset_index(drop=True), X_train_sm)
ols_model = ols_model_obj.fit() 
print("OLS-Modelltraining abgeschlossen.")

In [ ]:
# Funktion zur Ausgabe der Regressionskennzahlen
def regressionSummaryOLS(model):
    print(f"Anzahl der Datenpunkte: {model.nobs}")
    metrics = [
        ('SSR', model.ssr),
        ('Standardfehler', np.sqrt(model.mse_resid)),
        ('R2', model.rsquared),
        ('adj. R2', model.rsquared_adj),
        ('F_emp', model.fvalue),
        ('pValue', model.f_pvalue )
    ]

    # Formatierung der Ausgabe
    maxlength = max(len(m[0]) for m in metrics)
    fmt1 = f'{{:>{maxlength}}} : {{:.3f}}'
    print('Regressionskennzahlen\n')
    for metric, value in metrics:
        print(fmt1.format(metric, value))

regressionSummaryOLS(ols_model)

### Regressionskennzahlen der Vorhersage von Trainingsdaten

In [ ]:
def regressionSummary2(model, y_true, y_predicted):
    print(f"Anzahl gültiger Datenpunkte: {len(y_true)}\n")
    
    ssr = np.sum((y_true - y_predicted)**2)
    se = np.sqrt(ssr/model.df_resid)
    y_mean = np.mean(y_true)
    ess = np.sum((y_mean - y_predicted)**2)  
    sst = np.sum((y_mean - y_true)**2)
    R2 = ess / sst
    
    metrics = [
        ('SSR', ssr),
        ('Standardfehler', se),
        ('R2', R2)
    ]
    
    # Formatierung der Ausgabe
    maxlength = max(len(m[0]) for m in metrics)
    fmt1 = f'{{:>{maxlength}}} : {{:.4f}}'
    for metric, value in metrics:
        print(fmt1.format(metric, value))

print('\nOLS Regressionskennzahlen der Vorhersage von Trainingsdaten')
train_y_predict = ols_model.predict(X_train_sm)
regressionSummary2(ols_model, y_train, train_y_predict)
print("_" * 50)
print('\nOLS Regressionskennzahlen der Vorhersage von Testdaten')
test_y_predict = ols_model.predict(X_test_sm)
regressionSummary2(ols_model, y_test, test_y_predict)

In [ ]:
print(ols_model.summary())

### Interpretation der OLS-Ergebnisse:
#### Gesamtsignifikanz des Modells:
- F-Statistik: 74.09 
    - Dies ist ein sehr hoher Wert, der darauf hindeutet, dass das Modell als Ganzes statistisch signifikant ist. H0 wird eindeutig verworfen, da der p-Wert (1.24e-248) weit unter 0.05 liegt. 
#### Modellgüte:
- R-squared: 0.667: 
    - Ca. 66.7% der Varianz in der log-transformierten Like-Anzahl (`likes_log`) können durch die im Modell enthaltenen Features erklärt werden.
##### Wichtige numerische Features (standardisiert):
- `follower_count`: coef: 0.4502, P>|t|: 0.000
    - Eine Erhöhung des `follower_count` um eine Standardabweichung vom Mittelwert ist mit einem durchschnittlichen Anstieg von `likes_log` um ca. 0.4502 verbunden. Hochsignifikant.
- `hashtag_count`: coef: -0.7954, P>|t|: 0.000    
    - Eine Erhöhung des `hashtag_count` um eine Standardabweichung ist mit einem durchschnittlichen Rückgang von `likes_log` um ca. 0.7954 verbunden. Hochsignifikant und ein starker negativer Effekt.
##### Wichtige kategoriale Features (Dummy-Variablen):
- `is_professional_account_True`: coef: 4.0250, P>|t|: 0.000
    - Professionelle Accounts haben im Durchschnitt einen um ca. 4.03 höheren `likes_log`-Wert als nicht-professionelle Accounts (Referenzkategorie). Sehr starker positiver Prädiktor im Modell.
- `is_business_account_True`: coef: -0.3787, P>|t|: 0.024
    - Business Accounts haben im Durchschnitt einen um ca. 0.38 niedrigeren `likes_log`-Wert als Nicht-Business-Accounts (Referenzkategorie). Signifikanter negativer Effekt.
##### Kritische Nuance zur Interpretation der Koeffizientengrößen:
- Die Koeffizienten der standardisierten numerischen Variablen und der nicht-standardisierten Dummy-Variablen sind nicht direkt miteinander vergleichbar. Sie messen unterschiedliche Dinge. Die p-Werte (P>|t|) sind hier der bessere Indikator für die statistische Signifikanz des jeweiligen Einflusses. 
##### Diagnostik der Modelannahmen:
- Residuenverteilung: Prob(Omnibus): 0.000, Skew: -0.687, Kurtosis: 6.652
    - Kritisch: Die Residuen sind nicht normalverteilt. Dies ist eine Verletzung einer Kernannahme der OLS-Regression und bedeutet, dass die Standardfehler, t-Werte, p-Werte und Konfidenzintervalle der Koeffizienten nicht vollständig zuverlässig sind für die statistische Inferenz. 
- Multikollinearität: Cond. No.: 42.8.
    - Kritisch: Ein Wert über 30 deutet auf moderate bis starke Multikollinearität hin. Das bedeutet, dass einige deiner Prädiktorvariablen wahrscheinlich stark miteinander korrelieren.

### Diagramme des OLS-Modells

#### F-Verteilungs-Diagramm
Der F-Wert des Modells beträgt 74.09 (aus dem OLS-Summary).
Dieser Wert liegt weit im roten Bereich (deutlich rechts vom kritischen Wert)
Daher wird H₀ mit sehr hoher Sicherheit abgelehnt. Die Wahrscheinlichkeit, einen so hohen F-Wert zufällig zu beobachten, wenn tatsächlich kein Zusammenhang besteht, ist extrem gering (p ≈ 1.24e-248).

In [ ]:
alpha = 0.05

# Freiheitsgrade
df_1 = ols_model.df_model  # Anzahl der Regressoren
df_2 = ols_model.df_resid  # Freiheitsgrade der Residuen

# Kritischer F-Wert für das Modell
F_krit = stats.f.ppf(1 - alpha, df_1, df_2)

F_emp = 74.09  # empirischer F-Wert aus dem Modell

b1, b2 = 0, 3
x_v = np.arange(b1, b2, 0.01)
plt.figure(figsize=(8, 5))
plt.plot(x_v, stats.f.pdf(x_v, df_1, df_2))
fill_l = np.arange(F_krit, b2, (b2-F_krit)/100)
plt.fill_between(fill_l, stats.f.pdf(fill_l, df_1, df_2), color='red', alpha=0.5, label='Ablehnungsbereich H0')
plt.axvline(F_krit, color='green', linestyle='--', label=f'F_krit = {F_krit:.2f}')
plt.axvline(F_emp, color='blue', linestyle='-', label=f'F_emp = {F_emp:.2f}')
plt.axvspan(b1, 1.5,  alpha=0.2, label=f'H0 bestätigt (F < {F_krit:.2f})')
plt.legend()
plt.title(f'F-Verteilung \n Prädikatoren: df_1 = {int(df_1)}, Datenpunkte: df_2 = {int(df_2)}')
plt.xlabel('F-Wert', color ='green')
plt.ylabel('Dichte f(x)')
plt.xlim(b1, b2)
plt.show()

#### t-Verteilungs-Diagramm
- Die kritischen t-Werte bei α = 0.05 definieren den Ablehnungsbereich für die Nullhypothese. Werte außerhalb dieses Bereichs sind statistisch signifikant. Alle drei t-Werte liegen deutlich außerhalb des Nicht-Ablehnungsbereichs. 

In [ ]:
alpha = 0.05 # Signifikanzniveau
feature_name_follower = 'follower_count' 
feature_name_hashtag = 'hashtag_count' 
feature_name_account_category = 'account_category_Artist'

t_value_follower = ols_model.tvalues.get(feature_name_follower, np.nan) 
t_value_third_hashtag = ols_model.tvalues.get(feature_name_hashtag, np.nan)
t_value_account_category = ols_model.tvalues.get(feature_name_account_category, np.nan)

x_t = np.arange(-13, 13, 0.01) 
plt.figure(figsize=(8, 5))
plt.plot(x_t, stats.t.pdf(x_t, df=ols_model.df_resid))

# Kritische Werte für zweiseitigen Test
c_lower, c_upper = stats.t.interval(1 - alpha, df=ols_model.df_resid)
print(f'Ablehnungsbereich für H0: t < {c_lower:.3f} oder t > {c_upper:.3f}')

plt.axvline(c_lower, linestyle='--', color='red')
plt.axvline(c_upper, linestyle='--', color='red', label=f't_krit (Grenze) = {c_upper:.3f}')

# Empirische t-Werte
plt.axvline(t_value_follower - 0.1, linestyle='--', color='orange', label=f't(Follower) = {t_value_follower:.3f}')
plt.axvline(t_value_third_hashtag, linestyle='--', color='green', label=f't(Hashtag Count) = {t_value_third_hashtag:.3f}')
plt.axvline(t_value_account_category, linestyle='--', color='purple', label=f't(Account Category) = {t_value_account_category:.3f}')

plt.xlabel('t-Wert')
plt.ylabel('Dichtefunktion f(t)')
plt.title(f't-Verteilung und empirische t-Werte (df={ols_model.df_resid:.0f})')
plt.legend(loc=1)
plt.tight_layout()
plt.show()

#### Tukey-Anscombe-Plot
Der Tukey-Anscombe-Plot zeigt ein Muster, das auf nicht erfasste nicht-lineare Zusammenhänge hindeutet.

In [ ]:
fitted_vals = ols_model.fittedvalues
standardized_residuals = ols_model.resid / ols_model.resid.std()

plt.figure(figsize=(8, 6))
plt.axhline(y=0, color='red', linestyle='--')
plt.scatter(fitted_vals, standardized_residuals, alpha=0.5, edgecolors='k', s=50)
plt.title('Residuenplot - OLS')
plt.xlabel('Vorhergesagte Werte')
plt.ylabel('Residuen (Tatsächlich - Vorhergesagt)')
plt.grid(True)
plt.show()

In [ ]:
y_pred_test = ols_model.predict(X_test_sm)
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_test, alpha=0.5, edgecolors='k', s=50, label="Vorhersagen")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfekte Vorhersage')

plt.xlabel("Tatsächliche likes_log")
plt.ylabel("Vorhergesagte likes_log")
plt.title(f"Tatsächlich vs. Vorhergesagt - OLS (Testdaten) \nR²≈67%", color='darkblue')
plt.grid(True)
plt.legend()
plt.show()

#### Q-Q-Plot
Die S-Form im Q-Q-Plot zeigt Verletzungen der Normalverteilungsannahme.

In [ ]:
fig = qqplot(pp.scale(ols_model.resid), line='45', alpha=0.5)
ax = fig.gca()
for i in ax.get_children():
	if hasattr(i, 'set_markeredgecolor'):
		i.set_markeredgecolor('k')
	if hasattr(i, 'set_markersize'):
		i.set_markersize(6)
plt.title("Q-Q-Plot | Standardisierte Residuals")
plt.grid(True)
plt.show()

#### Histogramm der Residuen
- Die Verteilung ist spitzer (höhere Dichte um den Mittelpunkt).
- Das Histogramm zeigt eine höhere Konzentration in der Mitte als die Normalverteilungskurve vorhersagt.
- Die Randbereiche folgen nicht genau der erwarteten Normalverteilung.

In [ ]:
fig = plt.figure(figsize = (8,5))
bins = [i/2 for i in range(-4,5)]
# Daten werden zentriert und auf Einheitsvarianz skaliert
data = pp.scale(ols_model.resid)
plt.hist(data, bins = bins, rwidth = 0.9, density = True)
x = np.arange(-2.5, 2.5, 0.01)
plt.plot(x, stats.norm.pdf(x, loc = 0, scale = 1))
plt.title('Histogramm der Residuen')
plt.xlabel('Regression Standardisiertes Residuum')
plt.ylabel('Häufigkeit')
plt.grid(True)
plt.show()

In [ ]:
# Modelldaten in Pickle-Datei speichern
data_to_save = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'X': X,
    'y': y,
    'numerical_cols': numerical_cols,
    'categorical_cols': categorical_cols,
    'ols_model': ols_model 
}

with open('ergebnisse/modelle/ols_complete_data.pkl', 'wb') as f:
    pickle.dump(data_to_save, f)

### Fazit und Ausblick

Das Modell zeigt eine starke Gesamtsignifikanz, aber die nicht-normalverteilten Residuen schränken die Verlässlichkeit der statistischen Inferenz ein. Ein Random Forest könnte die Limitationen des OLS-Modells überwinden, insbesondere bei komplexen nicht-linearen Beziehungen und Interaktionen zwischen den Variablen.